Install and import libraries

In [1]:
!pip install -q gensim regex

import pandas as pd
import numpy as np
import regex as re

from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


Load dataset and recreate the official split

In [2]:
# Load TSV dataset
multilingual_df = pd.read_csv("GWSD_Multilingual.tsv", sep="\t")

# Create labelled dataset
score_columns = ["agree", "disagree", "neutral"]

labelled_df = multilingual_df.dropna(
    subset=score_columns
).copy()

labelled_df["stance"] = labelled_df[
    score_columns
].idxmax(axis=1)

labelled_df["stance"] = labelled_df["stance"].map({
    "agree": "Favour",
    "disagree": "Against",
    "neutral": "Neutral"
})

# Recreate official train/test split
train_df = labelled_df[
    labelled_df["in_held_out_test"] == False
].copy().reset_index(drop=True)

test_df = labelled_df[
    labelled_df["in_held_out_test"] == True
].copy().reset_index(drop=True)

# Target labels
y_train = train_df["stance"]
y_test = test_df["stance"]

print("Dataset shape:", multilingual_df.shape)
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Dataset shape: (2300, 20)
Training samples: 1850
Testing samples: 200

Training label distribution:
stance
Neutral    788
Favour     701
Against    361
Name: count, dtype: int64

Testing label distribution:
stance
Neutral    85
Favour     76
Against    39
Name: count, dtype: int64


Tokenize all three languages

In [3]:
def tokenize_multilingual(text):
    text = str(text).lower()
    tokens = re.findall(r'[\p{L}\p{M}\p{N}]+', text)
    return tokens


# Training sentences
sentences_en = [
    tokenize_multilingual(text)
    for text in train_df["sentence"]
]

sentences_hi = [
    tokenize_multilingual(text)
    for text in train_df["sentence_hi"]
]

sentences_bn = [
    tokenize_multilingual(text)
    for text in train_df["sentence_bn"]
]


# Testing sentences
test_sentences_en = [
    tokenize_multilingual(text)
    for text in test_df["sentence"]
]

test_sentences_hi = [
    tokenize_multilingual(text)
    for text in test_df["sentence_hi"]
]

test_sentences_bn = [
    tokenize_multilingual(text)
    for text in test_df["sentence_bn"]
]


print("English example:", sentences_en[0])
print("Hindi example:", sentences_hi[0])
print("Bengali example:", sentences_bn[0])

English example: ['warmer', 'than', 'normal', 'sea', 'surface', 'temperatures', 'are', 'a', 'key', 'player', 'in', 'the', 'development', 'of', 'hurricanes', 'such', 'as', 'katrina', 'and', 'superstorm', 'sandy', 'which', 'hit', 'the', 'u', 's', 'east', 'coast', 'in', '2011']
Hindi example: ['u', 's', 'पूर्वी', 'तट', 'पर', 'u', 's', 'पूर्वी', 'तट', 'मारा']
Bengali example: ['তাপমাত্রা', 'নাক্তিক', 'নাক্তিক', 'সাগর', 'পৃথিবীর', 'তাপমাত্রা', 'ক্যাট্রিনা', 'এবং', 'সুপার', 'স্টোরম', 'সান্ডি', 'जैसे', 'হুরগানগুলির', 'উন্নয়নে', 'একটি', 'গুরুত্বপূর্ণ', 'খেলা', 'হয়', 'যা', '২০১১', 'সালে', 'মার্কিন', 'পূর্ব', 'কাস্টমের', 'দিকে', 'হামলার']


Train three Word2Vec models

In [4]:
# English Word2Vec
word2vec_en = Word2Vec(
    sentences=sentences_en,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=42
)

# Hindi Word2Vec
word2vec_hi = Word2Vec(
    sentences=sentences_hi,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=42
)

# Bengali Word2Vec
word2vec_bn = Word2Vec(
    sentences=sentences_bn,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=30,
    seed=42
)

print("All three Word2Vec models trained successfully!")

All three Word2Vec models trained successfully!


Convert sentences into numerical vectors

In [5]:
def get_sentence_vectors(sentences, model):
    vectors = []

    for sentence in sentences:
        word_vectors = [
            model.wv[word]
            for word in sentence
            if word in model.wv
        ]

        if len(word_vectors) > 0:
            sentence_vector = np.mean(word_vectors, axis=0)
        else:
            sentence_vector = np.zeros(model.vector_size)

        vectors.append(sentence_vector)

    return np.array(vectors)


# Training sentence vectors
X_train_en_w2v = get_sentence_vectors(sentences_en, word2vec_en)
X_train_hi_w2v = get_sentence_vectors(sentences_hi, word2vec_hi)
X_train_bn_w2v = get_sentence_vectors(sentences_bn, word2vec_bn)


print("English shape:", X_train_en_w2v.shape)
print("Hindi shape:", X_train_hi_w2v.shape)
print("Bengali shape:", X_train_bn_w2v.shape)

English shape: (1850, 100)
Hindi shape: (1850, 100)
Bengali shape: (1850, 100)


Convert test sentences into Word2Vec vectors

In [6]:
# Convert test sentences into Word2Vec vectors

X_test_en_w2v = get_sentence_vectors(test_sentences_en, word2vec_en)

X_test_hi_w2v = get_sentence_vectors(test_sentences_hi, word2vec_hi)

X_test_bn_w2v = get_sentence_vectors(test_sentences_bn, word2vec_bn)


print("English test shape:", X_test_en_w2v.shape)
print("Hindi test shape:", X_test_hi_w2v.shape)
print("Bengali test shape:", X_test_bn_w2v.shape)

English test shape: (200, 100)
Hindi test shape: (200, 100)
Bengali test shape: (200, 100)


English – Logistic Regression

In [7]:
lr_en_w2v = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_en_w2v.fit(X_train_en_w2v, y_train)

y_pred_en_w2v = lr_en_w2v.predict(X_test_en_w2v)

print("English - Word2Vec + Logistic Regression")
print("----------------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_en_w2v))
print("Precision:", precision_score(y_test, y_pred_en_w2v, average="macro"))
print("Recall:", recall_score(y_test, y_pred_en_w2v, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred_en_w2v, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en_w2v))

English - Word2Vec + Logistic Regression
----------------------------------------
Accuracy: 0.595
Precision: 0.5909722222222221
Recall: 0.607361541107671
Macro F1: 0.5856102477691075

Classification Report:
              precision    recall  f1-score   support

     Against       0.41      0.67      0.50        39
      Favour       0.67      0.58      0.62        76
     Neutral       0.70      0.58      0.63        85

    accuracy                           0.59       200
   macro avg       0.59      0.61      0.59       200
weighted avg       0.63      0.59      0.60       200



Hindi – Logistic Regression

In [8]:
lr_hi_w2v = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_hi_w2v.fit(X_train_hi_w2v, y_train)

y_pred_hi_w2v = lr_hi_w2v.predict(X_test_hi_w2v)

print("Hindi - Word2Vec + Logistic Regression")
print("---------------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_hi_w2v))
print("Precision:", precision_score(y_test, y_pred_hi_w2v, average="macro"))
print("Recall:", recall_score(y_test, y_pred_hi_w2v, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred_hi_w2v, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hi_w2v))

Hindi - Word2Vec + Logistic Regression
---------------------------------------
Accuracy: 0.54
Precision: 0.5435561788997973
Recall: 0.5577412611468339
Macro F1: 0.5337048887773524

Classification Report:
              precision    recall  f1-score   support

     Against       0.36      0.64      0.46        39
      Favour       0.65      0.53      0.58        76
     Neutral       0.62      0.51      0.56        85

    accuracy                           0.54       200
   macro avg       0.54      0.56      0.53       200
weighted avg       0.58      0.54      0.55       200



Bengali – Logistic Regression

In [9]:
lr_bn_w2v = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_bn_w2v.fit(X_train_bn_w2v, y_train)

y_pred_bn_w2v = lr_bn_w2v.predict(X_test_bn_w2v)

print("Bengali - Word2Vec + Logistic Regression")
print("-----------------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_bn_w2v))
print("Precision:", precision_score(y_test, y_pred_bn_w2v, average="macro"))
print("Recall:", recall_score(y_test, y_pred_bn_w2v, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred_bn_w2v, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bn_w2v))

Bengali - Word2Vec + Logistic Regression
-----------------------------------------
Accuracy: 0.57
Precision: 0.5578377022038994
Recall: 0.574806170781403
Macro F1: 0.5612415906533553

Classification Report:
              precision    recall  f1-score   support

     Against       0.44      0.59      0.51        39
      Favour       0.60      0.61      0.60        76
     Neutral       0.63      0.53      0.58        85

    accuracy                           0.57       200
   macro avg       0.56      0.57      0.56       200
weighted avg       0.58      0.57      0.57       200



 English – Word2Vec + Linear SVM

In [10]:
svm_en_w2v = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_en_w2v.fit(X_train_en_w2v, y_train)

y_pred_en_svm_w2v = svm_en_w2v.predict(X_test_en_w2v)

print("English - Word2Vec + Linear SVM")
print("--------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_en_svm_w2v))
print("Precision:", precision_score(y_test, y_pred_en_svm_w2v, average="macro"))
print("Recall:", recall_score(y_test, y_pred_en_svm_w2v, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred_en_svm_w2v, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_en_svm_w2v))

English - Word2Vec + Linear SVM
--------------------------------
Accuracy: 0.595
Precision: 0.5783482926340069
Recall: 0.5902529702839301
Macro F1: 0.5811033008591276

Classification Report:
              precision    recall  f1-score   support

     Against       0.45      0.56      0.50        39
      Favour       0.61      0.62      0.61        76
     Neutral       0.68      0.59      0.63        85

    accuracy                           0.59       200
   macro avg       0.58      0.59      0.58       200
weighted avg       0.61      0.59      0.60       200



Hindi – Word2Vec + Linear SVM

In [11]:
svm_hi_w2v = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_hi_w2v.fit(X_train_hi_w2v, y_train)

y_pred_hi_svm_w2v = svm_hi_w2v.predict(X_test_hi_w2v)

print("Hindi - Word2Vec + Linear SVM")
print("------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_hi_svm_w2v))
print("Precision:", precision_score(y_test, y_pred_hi_svm_w2v, average="macro"))
print("Recall:", recall_score(y_test, y_pred_hi_svm_w2v, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred_hi_svm_w2v, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hi_svm_w2v))

Hindi - Word2Vec + Linear SVM
------------------------------
Accuracy: 0.595
Precision: 0.5700083542188805
Recall: 0.57128681432087
Macro F1: 0.5706156447969598

Classification Report:
              precision    recall  f1-score   support

     Against       0.45      0.46      0.46        39
      Favour       0.61      0.61      0.61        76
     Neutral       0.65      0.65      0.65        85

    accuracy                           0.59       200
   macro avg       0.57      0.57      0.57       200
weighted avg       0.60      0.59      0.60       200



Bengali – Word2Vec + Linear SVM

In [12]:
svm_bn_w2v = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_bn_w2v.fit(X_train_bn_w2v, y_train)

y_pred_bn_svm_w2v = svm_bn_w2v.predict(X_test_bn_w2v)

print("Bengali - Word2Vec + Linear SVM")
print("--------------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred_bn_svm_w2v))
print("Precision:", precision_score(y_test, y_pred_bn_svm_w2v, average="macro"))
print("Recall:", recall_score(y_test, y_pred_bn_svm_w2v, average="macro"))
print("Macro F1:", f1_score(y_test, y_pred_bn_svm_w2v, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bn_svm_w2v))

Bengali - Word2Vec + Linear SVM
--------------------------------
Accuracy: 0.59
Precision: 0.572003529337619
Recall: 0.5780093143870234
Macro F1: 0.5735193475352475

Classification Report:
              precision    recall  f1-score   support

     Against       0.47      0.51      0.49        39
      Favour       0.60      0.64      0.62        76
     Neutral       0.65      0.58      0.61        85

    accuracy                           0.59       200
   macro avg       0.57      0.58      0.57       200
weighted avg       0.60      0.59      0.59       200



Create the combined comparison of all 18 experiments

In [13]:
all_results = [
    # TF-IDF
    {"Language": "English", "Feature": "TF-IDF", "Model": "Logistic Regression", "Accuracy": 0.580, "Macro F1": 0.5843},
    {"Language": "English", "Feature": "TF-IDF", "Model": "Linear SVM", "Accuracy": 0.565, "Macro F1": 0.5594},
    {"Language": "Hindi", "Feature": "TF-IDF", "Model": "Logistic Regression", "Accuracy": 0.530, "Macro F1": 0.5203},
    {"Language": "Hindi", "Feature": "TF-IDF", "Model": "Linear SVM", "Accuracy": 0.525, "Macro F1": 0.5106},
    {"Language": "Bengali", "Feature": "TF-IDF", "Model": "Logistic Regression", "Accuracy": 0.565, "Macro F1": 0.5421},
    {"Language": "Bengali", "Feature": "TF-IDF", "Model": "Linear SVM", "Accuracy": 0.555, "Macro F1": 0.5237},

    # FastText
    {"Language": "English", "Feature": "FastText", "Model": "Logistic Regression", "Accuracy": 0.560, "Macro F1": 0.5535},
    {"Language": "English", "Feature": "FastText", "Model": "Linear SVM", "Accuracy": 0.615, "Macro F1": 0.6038},
    {"Language": "Hindi", "Feature": "FastText", "Model": "Logistic Regression", "Accuracy": 0.530, "Macro F1": 0.5258},
    {"Language": "Hindi", "Feature": "FastText", "Model": "Linear SVM", "Accuracy": 0.590, "Macro F1": 0.5703},
    {"Language": "Bengali", "Feature": "FastText", "Model": "Logistic Regression", "Accuracy": 0.605, "Macro F1": 0.5947},
    {"Language": "Bengali", "Feature": "FastText", "Model": "Linear SVM", "Accuracy": 0.635, "Macro F1": 0.6160},

    # Word2Vec
    {"Language": "English", "Feature": "Word2Vec", "Model": "Logistic Regression", "Accuracy": 0.595, "Macro F1": 0.5856},
    {"Language": "English", "Feature": "Word2Vec", "Model": "Linear SVM", "Accuracy": 0.595, "Macro F1": 0.5811},
    {"Language": "Hindi", "Feature": "Word2Vec", "Model": "Logistic Regression", "Accuracy": 0.540, "Macro F1": 0.5337},
    {"Language": "Hindi", "Feature": "Word2Vec", "Model": "Linear SVM", "Accuracy": 0.595, "Macro F1": 0.5706},
    {"Language": "Bengali", "Feature": "Word2Vec", "Model": "Logistic Regression", "Accuracy": 0.570, "Macro F1": 0.5612},
    {"Language": "Bengali", "Feature": "Word2Vec", "Model": "Linear SVM", "Accuracy": 0.590, "Macro F1": 0.5735}
]

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

results_df

,Language,Feature,Model,Accuracy,Macro F1
0,Bengali,FastText,Linear SVM,0.635,0.6160
1,English,FastText,Linear SVM,0.615,0.6038
2,Bengali,FastText,Logistic Regression,0.605,0.5947
3,English,Word2Vec,Logistic Regression,0.595,0.5856
4,English,TF-IDF,Logistic Regression,0.580,0.5843
5,English,Word2Vec,Linear SVM,0.595,0.5811
6,Bengali,Word2Vec,Linear SVM,0.590,0.5735
7,Hindi,Word2Vec,Linear SVM,0.595,0.5706
8,Hindi,FastText,Linear SVM,0.590,0.5703
9,Bengali,Word2Vec,Logistic Regression,0.570,0.5612


What have we learned?
1. FastText performs better overall than TF-IDF and Word2Vec in these experiments.
2. Linear SVM generally performs better than Logistic Regression with FastText.
3. Bengali has the highest baseline score, but we should not conclude that Bengali stance detection is inherently easier.
4. Against is generally the most difficult stance class.